[⬅ Back to the examples](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/try/)

# Push buttons

Machine panel push buttons. A plain button only sends commands; an illuminated
button also has a lamp, lit by the program to show a state (running, stopped,
fault to reset). Cap and lamp colors follow IEC 60073: green to start, red to
stop, amber for attention. **Run all the cells**, then press the buttons.

**References:** IEC 60073 (cap and lamp colors). These standards inspired
the widgets; the library does not claim conformity with them ([Standards and
references](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/standards/)).

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

In [ ]:
# JupyterLite: install the package (bundled with this site) in the browser kernel
%pip install -q anywidget-instruments

In [ ]:
import anywidget_instruments as ai

ai.theme_switch()  # light / system / dark theme (STYLE-008)

## Push buttons without a lamp

**START** and **STOP** *latch when released*: each press is kept until the
program reads it, so no press is lost. **JOG** is *switch until released*: the
motor turns only while you hold it (mouse, touch, or Space / Enter on the
focused button).

In [ ]:
start = ai.PushButton(text="START", color="green", label="Start")
stop = ai.PushButton(text="STOP", color="red", shape="round", label="Stop")
jog = ai.PushButton(
    text="JOG", color="black", mechanical_action="switch_until_released", label="Jog (hold)"
)
motor = ai.Motor(mode="indicator", label="Motor M1")
presses = ai.SevenSegment(0, digits=4, decimals=0, label="Start presses", size=(150, 60))
plain = {"run": False, "jog": False, "n": 0}


def show():
    motor.value = "forward" if plain["run"] or plain["jog"] else "stopped"


@start.on_change
def _(change):
    if start.read_latched():  # consume the press (BOOL-010)
        plain["run"] = True
        plain["n"] += 1
        presses.value = plain["n"]
        show()


@stop.on_change
def _(change):
    if stop.read_latched():
        plain["run"] = False
        show()


@jog.on_change
def _(change):
    plain["jog"] = bool(change["new"])
    show()


ai.Panel([start, stop, jog, motor, presses], columns=5)

## Illuminated push buttons

The lamps show the machine state: **RUN** is lit while the machine runs,
**STOP** while it is stopped. **FAULT** simulates a fault: the machine stops
and **RESET** flashes amber until you press it. Hold **LAMP TEST** to light
every lamp.

In [ ]:
run_pb = ai.PushButton(text="RUN", shape="round", lamp=False, lamp_color="green", label="Run")
halt_pb = ai.PushButton(text="STOP", shape="round", lamp=True, lamp_color="red", label="Halt")
reset_pb = ai.PushButton(text="RESET", lamp=False, lamp_color="amber", label="Reset")
fault_pb = ai.PushButton(text="FAULT", color="yellow", label="Simulate a fault")
test_pb = ai.PushButton(
    text="LAMP TEST", mechanical_action="switch_until_released", label="Lamp test"
)
machine = ai.Motor(mode="indicator", label="Motor M2")
lit = {"run": False, "fault": False, "test": False}


def lamps():
    test = lit["test"]
    run_pb.lamp = test or lit["run"]
    halt_pb.lamp = test or not lit["run"]
    reset_pb.lamp = test or lit["fault"]
    reset_pb.lamp_blink = lit["fault"] and not test
    machine.value = "fault" if lit["fault"] else ("forward" if lit["run"] else "stopped")


def on_press(button, action):
    @button.on_change
    def _(change):
        if button.read_latched():
            action()
            lamps()


def run():
    if not lit["fault"]:  # a fault must be reset before restarting
        lit["run"] = True


on_press(run_pb, run)
on_press(halt_pb, lambda: lit.update(run=False))
on_press(fault_pb, lambda: lit.update(run=False, fault=True))
on_press(reset_pb, lambda: lit.update(fault=False))


@test_pb.on_change
def _(change):
    lit["test"] = bool(change["new"])
    lamps()


ai.Panel([run_pb, halt_pb, reset_pb, machine, fault_pb, test_pb], columns=6)